# NHIS Mortality × Self-Reported Liver Disease — Pooled Analysis (2015–2018)

This notebook replicates the NHANES FIB-4 mortality analysis using NHIS data,
substituting **self-reported liver condition** (`LIVEV`) for the biomarker-derived
FIB-4 score.

**Key differences from the NHANES analysis:**

| | NHANES (notebook 03) | NHIS (this notebook) |
|---|---|---|
| Exposure | FIB-4 ≥2.67 (lab-derived) | Self-reported liver condition |
| N per year | ~5,000 | ~25–33,000 |
| Covariates | Measured BMI, BP, LDL-C, FPG | Self-reported BMI, diabetes, hypertension |
| Follow-up | PERMTH_EXM (exam date) | PERMTH_INT (interview date, quarter-level) |
| Mortality file | Same NDI linkage | Same NDI linkage |

**Progressive matching strategy:**

| Step | Covariates matched |
|------|-------------------|
| 0 | *(none — crude)* |
| 1 | Age, Sex |
| 2 | + BMI |
| 3 | + Smoking |
| 4 | + Diabetes, Hypertension |

# Can We Replicate This Analysis with NHIS Linked Mortality Data?

**Short answer: No** — not the FIB-4 analysis specifically. The NHIS lacks the
laboratory biomarkers required to compute FIB-4 and other non-invasive fibrosis
scores. But a complementary analysis using **self-reported liver disease** is
possible, and we built it in notebooks `04_nhis_data_download.ipynb` and
`05_nhis_pooled_analysis.ipynb`.

## What Our NHANES Analysis Requires

The pooled analysis in `03_pooled_analysis.ipynb` uses these variables:

| Variable | Purpose | Source in NHANES |
|---|---|---|
| AST, ALT | FIB-4 numerator/denominator | Lab (hepatic panel) |
| PLATELETS | FIB-4 denominator | Lab (CBC) |
| BMXBMI | Matching covariate, NFS input | Exam (anthropometry) |
| SBP_MEAN | Matching covariate | Exam (blood pressure) |
| LBDLDL | Step 5 covariate | Lab (lipids) |
| LBXGLU | Descriptive | Lab (plasma glucose) |
| SMOKE_EVER | Matching covariate | Questionnaire |
| RIDAGEYR, RIAGENDR | Matching + FIB-4 | Demographics |
| MORTSTAT, UCOD_LEADING, PERMTH_EXM | Outcome | Linked mortality file |

FIB-4 = (Age × AST) / (Platelets × √ALT). Without AST, ALT, and platelet
count the score cannot be computed.

## What NHIS Provides

The NHIS is a household interview survey. It collects no blood samples and
performs no physical examinations. Its strengths are large sample size
(~35,000–40,000 adults/year) and long time span (1957–present, with
public-use linked mortality files for 1986–2018, follow-up through 2019).

### Variables NHIS *does* collect annually

- **Demographics**: age, sex, race/ethnicity, education, income
- **BMI**: self-reported height and weight (available 1976+; sample-adult
  self-report only from 1997+)
- **Diabetes**: "Ever told by a doctor you have diabetes" (annually)
- **Smoking**: lifetime cigarette use, current status (annually)
- **Hypertension**: "Ever told you have hypertension" (annually)
- **Alcohol use**: frequency and quantity (most years)
- **Health care access/utilization**: insurance, usual source of care, ER visits

### Variables NHIS *does not* collect

- AST, ALT, or any liver enzyme
- Platelet count or any CBC component
- Albumin, LDL cholesterol, glucose, HbA1c
- Blood pressure measurement (self-report of hypertension diagnosis only)
- Any physical examination measurement

### Liver-specific questions in NHIS

- **Pre-2019**: `LIVERFATYRC` — chronic fatty liver in the past year
  (constructed from condition records by IPUMS NHIS)
- **2019+ redesign**: "Ever told by a doctor you have cirrhosis or any other
  kind of long-term liver condition?" — part of the annual chronic-conditions
  battery (rotating content)

These are self-reported diagnoses, not biomarker-derived classifications.

## NHIS Linked Mortality Files

NHIS data *are* linked to the National Death Index, with the same structure as
the NHANES linked mortality files:

- **Public-use**: NHIS 1986–2018, follow-up through December 31, 2019
- **Restricted-use**: NHIS 1986–2021, follow-up through December 31, 2022
- Variables: `ELIGSTAT`, `MORTSTAT`, `UCOD_LEADING` (113-group recode),
  `PERMTH_INT`, `DIABETES` (MCOD flag), `HYPERTEN` (MCOD flag)
- The same data-perturbation approach used for NHANES public-use files applies

Source: [CDC NCHS Public-Use Linked Mortality Files](https://www.cdc.gov/nchs/data-linkage/mortality-public.htm)

## Why a Direct Replication Is Not Feasible

The fundamental mismatch: our analysis defines exposure using a *biomarker-derived*
fibrosis score (FIB-4 ≥ 2.67). NHIS has no biomarkers. There is no way to
compute FIB-4, NFS, APRI, or any other non-invasive fibrosis index from NHIS
data alone.

There is also no NHIS–NHANES cross-linkage. The two surveys draw from separate
sampling frames, and participant identifiers are not shared.

## What *Could* Be Done with NHIS

Although we cannot replicate the FIB-4 analysis, NHIS offers a complementary
angle:

### 1. Self-reported liver disease and mortality

- **Exposure**: Self-reported diagnosis of liver condition (cirrhosis /
  long-term liver disease, 2019+ annual core; fatty liver, pre-1997 condition
  records)
- **Outcome**: All-cause and cause-specific mortality via linked NDI
- **Matching**: Age, sex, race, BMI (self-reported), diabetes, smoking,
  hypertension — all available
- **Advantage**: Massive sample size (~35k adults/year × 30+ years) and longer
  follow-up windows than NHANES offers
- **Limitation**: Self-reported liver disease captures only *diagnosed* cases,
  missing the large pool of undiagnosed NAFLD/fibrosis that the biomarker
  approach identifies

### 2. Diabetes/obesity subgroup mortality trends

NHIS data could support a time-trends analysis of mortality among adults with
diabetes and obesity (major NAFLD risk factors), matched to controls, using the
same propensity-score matching framework we built for NHANES. This would not
measure fibrosis directly but would characterize the mortality burden in the
population most at risk for it.

### 3. Validating the self-report question against NHANES

One useful cross-survey analysis: compare the prevalence of self-reported liver
disease in NHIS with the biomarker-derived fibrosis prevalence in NHANES for
overlapping years. This cannot use linked individual records (the surveys are
independent) but could compare population-level estimates and assess how much
subclinical fibrosis the self-report question misses.

## Summary

| Feature | NHANES (current analysis) | NHIS (potential) |
|---|---|---|
| Lab biomarkers (AST, ALT, platelets) | Yes | **No** |
| FIB-4 / NFS computable | Yes | **No** |
| Self-reported liver disease | Yes | Yes |
| Linked mortality (public-use) | 1999–2018 | 1986–2018 |
| Sample size per year | ~5,000 adults | ~35,000 adults |
| Physical exam (BMI, BP) | Measured | Self-reported |
| Demographics, smoking, diabetes | Yes | Yes |

**Bottom line**: NHIS linked mortality data cannot replicate the FIB-4–based
analysis. NHIS could support a *complementary* study of self-reported liver
disease and mortality, with the advantage of much larger sample sizes and
longer follow-up, but the disadvantage of capturing only diagnosed disease.


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from IPython.display import Markdown, display

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (9, 5)
sns.set_style('whitegrid')

## Configuration

In [ ]:
DERIVED = os.path.abspath(os.path.join('..', 'data', 'nhis', 'derived'))

WINDOW_24 = 24
WINDOW_36 = 36

# UCOD coding in the NHIS linked mortality file uses the same 10-group recode,
# but 2015–2018 files use a coarsened version (only codes 1, 2, 3, 4, 5, 10).
UCOD_LABELS = {
    1: 'Heart disease', 2: 'Malignant neoplasms', 3: 'Chronic lower resp.',
    4: 'Accidents', 5: 'Cerebrovascular', 6: "Alzheimer's", 7: 'Diabetes',
    8: 'Influenza/pneumonia', 9: 'Nephritis', 10: 'All other causes',
}

MATCH_STEPS = [
    {'label': 'Step 1: Demographics',     'covariates': ['AGE', 'FEMALE']},
    {'label': 'Step 2: + BMI',            'covariates': ['AGE', 'FEMALE', 'BMI']},
    {'label': 'Step 3: + Smoking',        'covariates': ['AGE', 'FEMALE', 'BMI', 'SMOKE_EVER']},
    {'label': 'Step 4: + Diab, Hyp',      'covariates': ['AGE', 'FEMALE', 'BMI', 'SMOKE_EVER', 'DIABETES', 'HYPERTENSION']},
]

## Load pooled data

In [ ]:
pooled = pd.read_parquet(os.path.join(DERIVED, 'nhis_2015_2018_pooled.parquet'))
print(f'Pooled: {len(pooled):,} adults from {pooled["YEAR"].nunique()} years '
      f'({sorted(pooled["YEAR"].unique())})')
print(f'Deaths: {(pooled["MORTSTAT"]==1).sum():,}')
print(f'PERMTH_INT range: {pooled["PERMTH_INT"].min():.0f}\u2013{pooled["PERMTH_INT"].max():.0f}')
print(f'Liver+ (self-report): {(pooled["LIVER_EVER"]==1).sum():,}')
print(f'Liver\u2212 (self-report): {(pooled["LIVER_EVER"]==0).sum():,}')

## Define outcomes

In [ ]:
def add_outcomes(df, window):
    w = f'_{window}m'
    df[f'FU{w}']    = df['PERMTH_INT'].clip(upper=window)
    df[f'PY{w}']    = df[f'FU{w}'] / 12.0
    df[f'DEATH{w}'] = ((df['MORTSTAT']==1) & (df['PERMTH_INT']<=window)).astype(int)
    for code in UCOD_LABELS:
        df[f'D_UCOD{code}{w}'] = ((df[f'DEATH{w}']==1) & (df['UCOD_LEADING']==code)).astype(int)
    return df

pooled = add_outcomes(pooled, WINDOW_24)
pooled = add_outcomes(pooled, WINDOW_36)

for w in [24, 36]:
    wname = f'_{w}m'
    d = pooled[f'DEATH{wname}'].sum()
    pct = (pooled['PERMTH_INT']>=w).mean()*100
    print(f'{w}m window: {d:,} deaths, {pct:.0f}% with >={w}m FU')

## Descriptive table

In [ ]:
rows = []
for year in sorted(pooled['YEAR'].unique()):
    sub = pooled[pooled['YEAR']==year]
    rows.append({
        'Year': int(year),
        'N': len(sub),
        'Liver+': int((sub['LIVER_EVER']==1).sum()),
        'Liver\u2212': int((sub['LIVER_EVER']==0).sum()),
        'Deaths 24m': int(sub['DEATH_24m'].sum()),
        'Deaths 36m': int(sub['DEATH_36m'].sum()),
        '% \u226524m FU': f"{(sub['PERMTH_INT']>=24).mean()*100:.0f}%",
        '% \u226536m FU': f"{(sub['PERMTH_INT']>=36).mean()*100:.0f}%",
        'Max FU (m)': int(sub['PERMTH_INT'].max()),
    })

rows.append({
    'Year': 'TOTAL',
    'N': len(pooled),
    'Liver+': int((pooled['LIVER_EVER']==1).sum()),
    'Liver\u2212': int((pooled['LIVER_EVER']==0).sum()),
    'Deaths 24m': int(pooled['DEATH_24m'].sum()),
    'Deaths 36m': int(pooled['DEATH_36m'].sum()),
    '% \u226524m FU': f"{(pooled['PERMTH_INT']>=24).mean()*100:.0f}%",
    '% \u226536m FU': f"{(pooled['PERMTH_INT']>=36).mean()*100:.0f}%",
    'Max FU (m)': int(pooled['PERMTH_INT'].max()),
})

desc_df = pd.DataFrame(rows)
desc_df

## Unmatched death rates

In [ ]:
fib_col = 'LIVER_EVER'

def rate_table(df, fib_col, window, label=''):
    w = f'_{window}m'
    rows = []
    for val, name in [(1,'Liver+'),(0,'Liver\u2212')]:
        s = df[df[fib_col]==val]
        n = len(s)
        if n == 0: continue
        d = s[f'DEATH{w}'].sum()
        py = s[f'PY{w}'].sum()
        row = {'Label': label, 'Window': window, 'Group': name,
               'N': n, 'Deaths': int(d), 'PY': round(py,1),
               'Risk': round(d/n, 4) if n else np.nan,
               'Rate/1000PY': round(d/py*1000, 1) if py>0 else np.nan}
        for code, lab in UCOD_LABELS.items():
            row[f'd_{lab}'] = int(s[f'D_UCOD{code}{w}'].sum())
        rows.append(row)
    return pd.DataFrame(rows)

rates_24 = rate_table(pooled, fib_col, 24, 'Pooled')
rates_36 = rate_table(pooled, fib_col, 36, 'Pooled')
pooled_rates = pd.concat([rates_24, rates_36], ignore_index=True)
pooled_rates[['Label','Window','Group','N','Deaths','PY','Risk','Rate/1000PY']]

In [ ]:
# Per-year unmatched rates
year_rates = []
for year in sorted(pooled['YEAR'].unique()):
    sub = pooled[pooled['YEAR']==year]
    rt = rate_table(sub, fib_col, 24, str(int(year)))
    year_rates.append(rt)

year_rates_df = pd.concat(year_rates, ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 5))
years_list = sorted(pooled['YEAR'].unique())
x = np.arange(len(years_list))
w = 0.35
for i, (grp, color) in enumerate([('Liver\u2212','#2ca02c'),('Liver+','#d62728')]):
    vals = []
    for yr in years_list:
        row = year_rates_df[(year_rates_df['Label']==str(int(yr))) & (year_rates_df['Group']==grp)]
        vals.append(row.iloc[0]['Rate/1000PY'] if len(row) else 0)
    offset = -w/2 + i*w
    ax.bar(x+offset, vals, w, label=grp, color=color, edgecolor='black', lw=0.5, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([str(int(y)) for y in years_list])
ax.set_ylabel('Death rate per 1,000 PY')
ax.set_title('NHIS: All-cause 24m mortality by self-reported liver condition \u2014 per year')
ax.legend(); ax.set_ylim(bottom=0)
plt.tight_layout(); plt.show()

## Matching and analysis functions

In [ ]:
def propensity_match(df, fib_col, covariates, caliper=0.2, random_state=42):
    sub = df.dropna(subset=[fib_col] + covariates).copy()
    treated  = sub[sub[fib_col]==1]
    control  = sub[sub[fib_col]==0]
    if len(treated) < 5 or len(control) < 5:
        return pd.DataFrame(), len(treated), len(control)
    
    X = sub[covariates].copy()
    for c in covariates:
        if sub[c].nunique() > 2:
            X[c] = (X[c] - X[c].mean()) / (X[c].std() + 1e-8)
    y = sub[fib_col].astype(int)
    try:
        model = sm.Logit(y, sm.add_constant(X)).fit(disp=0, maxiter=100)
        sub['ps'] = model.predict(sm.add_constant(X))
    except Exception as e:
        print(f'  PS model failed: {e}')
        return pd.DataFrame(), len(treated), len(control)
    
    cal = caliper * sub['ps'].std()
    treated_idx = sub[sub[fib_col]==1].index.tolist()
    control_idx = sub[sub[fib_col]==0].index.tolist()
    rng = np.random.default_rng(random_state)
    rng.shuffle(treated_idx)
    
    matched_t, matched_c = [], []
    used = set()
    ctrl_ps = sub.loc[control_idx, 'ps'].values
    ctrl_arr = np.array(control_idx)
    for t_i in treated_idx:
        dists = np.abs(ctrl_ps - sub.loc[t_i, 'ps'])
        mask = np.array([c not in used for c in ctrl_arr])
        dists[~mask] = np.inf
        best = np.argmin(dists)
        if dists[best] <= cal:
            matched_t.append(t_i)
            matched_c.append(ctrl_arr[best])
            used.add(ctrl_arr[best])
    
    if not matched_t:
        return pd.DataFrame(), len(treated), len(control)
    
    mt = sub.loc[matched_t].copy(); mt['MATCH_ID'] = range(len(matched_t))
    mc = sub.loc[matched_c].copy(); mc['MATCH_ID'] = range(len(matched_c))
    return pd.concat([mt, mc], ignore_index=True), len(treated), len(control)


def covariate_balance(df, fib_col, covariates):
    rows = []
    for c in covariates:
        t = df.loc[df[fib_col]==1, c].dropna()
        ctrl = df.loc[df[fib_col]==0, c].dropna()
        if len(t)==0 or len(ctrl)==0:
            rows.append({'Covariate': c, 'SMD': np.nan})
            continue
        pooled_std = np.sqrt((t.var() + ctrl.var()) / 2)
        smd = (t.mean() - ctrl.mean()) / pooled_std if pooled_std > 0 else 0
        rows.append({'Covariate': c, 'SMD': round(smd, 3)})
    return pd.DataFrame(rows)


def risk_ratio(df, fib_col, window):
    w = f'_{window}m'
    t = df[df[fib_col]==1]; c = df[df[fib_col]==0]
    n1, n0 = len(t), len(c)
    d1 = t[f'DEATH{w}'].sum(); d0 = c[f'DEATH{w}'].sum()
    if n1==0 or n0==0 or d0==0:
        return {'RR':np.nan,'lo':np.nan,'hi':np.nan,'d1':int(d1),'d0':int(d0),'n1':n1,'n0':n0}
    r1, r0 = d1/n1, d0/n0
    rr = r1/r0
    se = np.sqrt(1/d1 - 1/n1 + 1/d0 - 1/n0) if d1>0 else np.nan
    lo = np.exp(np.log(rr)-1.96*se) if np.isfinite(se) else np.nan
    hi = np.exp(np.log(rr)+1.96*se) if np.isfinite(se) else np.nan
    return {'RR':round(rr,2),'lo':round(lo,2),'hi':round(hi,2),
            'd1':int(d1),'d0':int(d0),'n1':n1,'n0':n0}


def plot_km(df, fib_col, window, title_extra=''):
    w = f'_{window}m'
    sub = df[df[fib_col].notna()].copy()
    sub['T'] = sub[f'FU{w}']
    sub['E'] = sub[f'DEATH{w}']
    fig, ax = plt.subplots(figsize=(7, 5))
    kmf = KaplanMeierFitter()
    colors = {1: '#d62728', 0: '#2ca02c'}
    labels = {1: 'Liver+', 0: 'Liver\u2212'}
    for val in [0, 1]:
        grp = sub[sub[fib_col] == val]
        if len(grp) == 0: continue
        kmf.fit(grp['T'], grp['E'],
                label=f"{labels[val]} (n={len(grp):,}, d={int(grp['E'].sum()):,})")
        kmf.plot_survival_function(ax=ax, color=colors[val], ci_show=True, ci_alpha=0.15)
    g1 = sub[sub[fib_col]==1]; g0 = sub[sub[fib_col]==0]
    if len(g1)>0 and len(g0)>0 and (g1['E'].sum()+g0['E'].sum())>0:
        lr = logrank_test(g1['T'], g0['T'], g1['E'], g0['E'])
        pstr = 'p < 0.001' if lr.p_value < 0.001 else f'p = {lr.p_value:.3f}'
        ax.text(0.98, 0.02, f'Log-rank {pstr}', transform=ax.transAxes,
                ha='right', va='bottom', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5))
    ax.set_xlabel('Months from interview')
    ax.set_ylabel('Survival probability')
    ax.set_title(f'NHIS 2015\u20132018: Liver+ vs Liver\u2212{title_extra}\n({window}m window)')
    ax.set_xlim(0, window)
    ax.set_ylim(bottom=max(0, ax.get_ylim()[0] - 0.02))
    ax.legend(loc='lower left', fontsize=9)
    plt.tight_layout(); plt.show()

## Unmatched Kaplan-Meier curves

In [ ]:
plot_km(pooled, fib_col, 24, ' \u2014 unmatched')
plot_km(pooled, fib_col, 36, ' \u2014 unmatched')

## Progressive propensity-score matching

In [ ]:
all_covs = ['AGE', 'FEMALE', 'BMI', 'SMOKE_EVER', 'DIABETES', 'HYPERTENSION']

prog_results = {}

# Step 0: Crude
sub0 = pooled[pooled[fib_col].notna()].copy()
rr24_crude = risk_ratio(sub0, fib_col, 24)
rr36_crude = risk_ratio(sub0, fib_col, 36)
prog_results['Step 0: Crude'] = {
    'matched': sub0, 'rr_24': rr24_crude, 'rr_36': rr36_crude,
    'n_pairs': None, 'covariates': [],
    'n_treated': int((sub0[fib_col]==1).sum()),
    'n_control': int((sub0[fib_col]==0).sum()),
}
print(f'Step 0 (Crude): n+={rr24_crude["n1"]:,}, n\u2212={rr24_crude["n0"]:,}')
print(f'  24m: d+={rr24_crude["d1"]}, d\u2212={rr24_crude["d0"]}, RR={rr24_crude["RR"]} [{rr24_crude["lo"]}\u2013{rr24_crude["hi"]}]')
print(f'  36m: d+={rr36_crude["d1"]}, d\u2212={rr36_crude["d0"]}, RR={rr36_crude["RR"]} [{rr36_crude["lo"]}\u2013{rr36_crude["hi"]}]')

# Steps 1-4
for step in MATCH_STEPS:
    covs = step['covariates']
    label = step['label']
    print(f'\n{label} \u2014 covariates: {covs}')
    
    mdf, n_t, n_c = propensity_match(pooled, fib_col, covs)
    if len(mdf) == 0:
        print('  No matches')
        prog_results[label] = {
            'matched': pd.DataFrame(), 'rr_24': None, 'rr_36': None,
            'n_pairs': 0, 'covariates': covs, 'n_treated': n_t, 'n_control': n_c,
        }
        continue
    
    mdf = add_outcomes(mdf, 24)
    mdf = add_outcomes(mdf, 36)
    n_pairs = int((mdf[fib_col]==1).sum())
    rr24 = risk_ratio(mdf, fib_col, 24)
    rr36 = risk_ratio(mdf, fib_col, 36)
    
    prog_results[label] = {
        'matched': mdf, 'rr_24': rr24, 'rr_36': rr36,
        'n_pairs': n_pairs, 'covariates': covs,
        'n_treated': n_t, 'n_control': n_c,
    }
    
    print(f'  Available: treated={n_t:,}, control={n_c:,}')
    print(f'  Matched {n_pairs:,} pairs')
    print(f'  24m: d+={rr24["d1"]}, d\u2212={rr24["d0"]}, RR={rr24["RR"]} [{rr24["lo"]}\u2013{rr24["hi"]}]')
    print(f'  36m: d+={rr36["d1"]}, d\u2212={rr36["d0"]}, RR={rr36["RR"]} [{rr36["lo"]}\u2013{rr36["hi"]}]')
    
    bal = covariate_balance(mdf, fib_col, all_covs)
    print(f'  Balance: ' + ', '.join(f'{r["Covariate"]}={abs(r["SMD"]):.2f}'
          for _, r in bal.iterrows() if pd.notna(r['SMD'])))

### Progressive matching summary table

In [ ]:
step_order = ['Step 0: Crude'] + [s['label'] for s in MATCH_STEPS]
summary_rows = []

for label in step_order:
    res = prog_results[label]
    row = {
        'Step': label,
        'Covariates': ', '.join(res['covariates']) if res['covariates'] else '(none)',
        'N+ avail': res['n_treated'],
        'N\u2212 avail': res['n_control'],
    }
    if res['n_pairs'] is not None:
        row['N matched pairs'] = res['n_pairs']
    else:
        row['N matched pairs'] = f"{res['n_treated']} vs {res['n_control']}"
    
    for w, rr_key in [(24, 'rr_24'), (36, 'rr_36')]:
        rr = res[rr_key]
        if rr and pd.notna(rr['RR']):
            row[f'd+({w}m)'] = rr['d1']
            row[f'd\u2212({w}m)'] = rr['d0']
            row[f'RR({w}m)'] = rr['RR']
            row[f'95%CI({w}m)'] = f"{rr['lo']}\u2013{rr['hi']}"
        else:
            row[f'd+({w}m)'] = '\u2014'
            row[f'd\u2212({w}m)'] = '\u2014'
            row[f'RR({w}m)'] = '\u2014'
            row[f'95%CI({w}m)'] = '\u2014'
    summary_rows.append(row)

prog_summary = pd.DataFrame(summary_rows)
prog_summary

### Forest plot

In [ ]:
for window in [24, 36]:
    rr_key = f'rr_{window}'
    fig, ax = plt.subplots(figsize=(9, 5))
    y_labels, rrs, los, his, annotations = [], [], [], [], []
    
    for label in step_order:
        res = prog_results[label]
        rr = res[rr_key]
        if rr is None or pd.isna(rr['RR']):
            continue
        y_labels.append(label)
        rrs.append(rr['RR'])
        los.append(rr['lo'])
        his.append(rr['hi'])
        n = res['n_pairs']
        n_str = f'n={n:,} pairs' if n is not None else f"n={res['n_treated']:,}+{res['n_control']:,}"
        annotations.append(f"RR={rr['RR']:.1f} [{rr['lo']:.1f}\u2013{rr['hi']:.1f}]  ({n_str})")
    
    y_pos = np.arange(len(y_labels))
    ax.errorbar(rrs, y_pos,
                xerr=[np.array(rrs)-np.array(los), np.array(his)-np.array(rrs)],
                fmt='o', color='#1f77b4', capsize=4, markersize=8, lw=1.5)
    ax.axvline(1.0, color='gray', ls='--', lw=0.8)
    
    x_right = max(his) * 1.05
    for i, ann in enumerate(annotations):
        ax.text(x_right, i, ann, va='center', fontsize=8)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel(f'Risk Ratio ({window}-month mortality)')
    ax.set_title(f'NHIS 2015\u20132018: Progressive matching \u2014 Liver+ vs Liver\u2212\n({window}-month window)')
    ax.set_xlim(left=0)
    plt.tight_layout(); plt.show()

### Love plot: covariate balance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

pre_df = pooled[pooled[fib_col].notna()]
pre_bal = covariate_balance(pre_df, fib_col, all_covs)
y = np.arange(len(all_covs))
ax.scatter(pre_bal['SMD'].abs(), y, marker='x', color='gray', s=50,
           label='Unmatched', zorder=3)

colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(MATCH_STEPS)))
markers = ['o', 's', 'D', '^']

for i, step in enumerate(MATCH_STEPS):
    res = prog_results[step['label']]
    if len(res['matched']) == 0:
        continue
    bal = covariate_balance(res['matched'], fib_col, all_covs)
    ax.scatter(bal['SMD'].abs(), y, marker=markers[i],
               color=colors[i], s=50, label=step['label'], zorder=3)

ax.axvline(0.1, color='red', ls='--', lw=0.8, label='|SMD|=0.1')
ax.set_yticks(y)
ax.set_yticklabels(all_covs)
ax.set_xlabel('|Standardized Mean Difference|')
ax.set_title('NHIS: covariate balance across progressive matching steps')
ax.legend(loc='lower right', fontsize=7, ncol=2)
ax.set_xlim(left=0)
plt.tight_layout(); plt.show()

### KM curves at each progressive matching step

In [ ]:
for label in step_order:
    res = prog_results[label]
    if len(res['matched']) == 0:
        continue
    suffix = ' \u2014 unmatched' if label == 'Step 0: Crude' else f' \u2014 {label}'
    plot_km(res['matched'], fib_col, 36, suffix)

## Cause-specific mortality (unmatched)

In [ ]:
rt = rate_table(pooled, fib_col, 36, 'Pooled 2015\u20132018')

dcols = [c for c in rt.columns if c.startswith('d_')]
active = [c for c in dcols if rt[c].sum()>0]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(active))
w = 0.35
for i, (grp, color) in enumerate([('Liver\u2212','#2ca02c'),('Liver+','#d62728')]):
    row = rt[rt['Group']==grp]
    if len(row)==0: continue
    row = row.iloc[0]
    py = row['PY']
    vals = [row[c]/py*1000 if py>0 else 0 for c in active]
    cnts = [int(row[c]) for c in active]
    offset = -w/2 + i*w
    ax.bar(x+offset, vals, w, label=grp, color=color,
           edgecolor='black', lw=0.5, alpha=0.8)
    for j, (v, cnt) in enumerate(zip(vals, cnts)):
        if cnt>0:
            ax.text(x[j]+offset, v+0.3, str(cnt), ha='center', va='bottom', fontsize=7)

labels = [c.replace('d_','') for c in active]
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Rate per 1,000 PY')
ax.set_title('NHIS 2015\u20132018: cause-group mortality (36m, unmatched)')
ax.legend(fontsize=8); ax.set_ylim(bottom=0)
plt.tight_layout(); plt.show()

## Cause-specific mortality: matched KM curves

After PS-matching on age, sex, BMI, smoking, diabetes, and hypertension (Step 4),
compare cause-specific survival.

In [ ]:
# Use Step 4 matched data
step4_res = prog_results.get('Step 4: + Diab, Hyp', {})
mdf_cause = step4_res.get('matched', pd.DataFrame())

if len(mdf_cause) > 0:
    n_pairs_cause = int((mdf_cause[fib_col]==1).sum())
    
    # Balance
    bal = covariate_balance(mdf_cause, fib_col, all_covs)
    print(f'Matched {n_pairs_cause:,} pairs')
    print('\nPost-match balance:')
    for _, r in bal.iterrows():
        smd = r['SMD']
        if pd.notna(smd):
            flag = ' *' if abs(smd) > 0.1 else ''
            print(f"  {r['Covariate']:14s} SMD = {smd:+.3f}{flag}")
    
    # Cause table
    cause_rows = []
    for code, label in UCOD_LABELS.items():
        col = f'D_UCOD{code}_36m'
        if col not in mdf_cause.columns:
            continue
        d_plus = int(mdf_cause.loc[mdf_cause[fib_col]==1, col].sum())
        d_minus = int(mdf_cause.loc[mdf_cause[fib_col]==0, col].sum())
        cause_rows.append({'UCOD': code, 'Cause': label,
                           'd(Liver+)': d_plus, 'd(Liver\u2212)': d_minus,
                           'Total': d_plus + d_minus})
    
    cause_df = pd.DataFrame(cause_rows).sort_values('Total', ascending=False)
    print(f'\nCause-specific deaths within 36 months (matched sample):')
    display(cause_df[cause_df['Total'] > 0])
else:
    print('No matched data for cause-specific analysis.')

In [ ]:
if len(mdf_cause) > 0:
    MIN_EVENTS = 5
    plot_causes = cause_df[cause_df['Total'] >= MIN_EVENTS].sort_values('Total', ascending=False)
    n_plot = len(plot_causes)
    
    if n_plot > 0:
        ncols = min(3, n_plot)
        nrows = (n_plot + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4.5*nrows))
        if n_plot == 1:
            axes = np.array([axes])
        axes = np.array(axes).flatten()
        
        group_labels = {1: 'Liver+', 0: 'Liver\u2212'}
        cause_colors = {1: '#d62728', 0: '#2ca02c'}
        
        for i, (_, crow) in enumerate(plot_causes.iterrows()):
            ax = axes[i]
            code = crow['UCOD']
            sub = mdf_cause[mdf_cause[fib_col].notna()].copy()
            sub['T'] = sub['FU_36m']
            sub['E'] = sub[f'D_UCOD{code}_36m']
            
            kmf = KaplanMeierFitter()
            for val in [0, 1]:
                grp = sub[sub[fib_col] == val]
                if len(grp) == 0: continue
                d = int(grp['E'].sum())
                kmf.fit(grp['T'], grp['E'], label=f"{group_labels[val]} (d={d})")
                kmf.plot_survival_function(ax=ax, color=cause_colors[val],
                                           ci_show=True, ci_alpha=0.15)
            
            g1 = sub[sub[fib_col]==1]; g0 = sub[sub[fib_col]==0]
            if g1['E'].sum() + g0['E'].sum() > 0:
                lr = logrank_test(g1['T'], g0['T'], g1['E'], g0['E'])
                pstr = 'p < 0.001' if lr.p_value < 0.001 else f'p = {lr.p_value:.3f}'
                ax.text(0.98, 0.02, f'Log-rank {pstr}', transform=ax.transAxes,
                        ha='right', va='bottom', fontsize=8,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5))
            
            ax.set_xlabel('Months')
            ax.set_ylabel('Survival prob.')
            ax.set_title(crow['Cause'], fontsize=10)
            ax.set_xlim(0, 36)
            ax.legend(loc='lower left', fontsize=7)
        
        for j in range(i+1, len(axes)):
            axes[j].set_visible(False)
        
        fig.suptitle(
            f'Cause-specific survival: Liver+ vs Liver\u2212 (PS-matched)\n'
            f'Matched on age, sex, BMI, smoking, diabetes, hypertension \u2014 '
            f'NHIS 2015\u20132018, 36m, {n_pairs_cause:,} pairs',
            fontsize=11, y=1.02)
        plt.tight_layout()
        plt.show()

## Summary

In [ ]:
en = '\u2013'
dash = '\u2014'
times = '\u00d7'

def fmt_rr(rr, default=dash):
    if not rr or not isinstance(rr, dict):
        return default
    rr_val = rr.get('RR', dash)
    lo = rr.get('lo', '?')
    hi = rr.get('hi', '?')
    return f"{rr_val} [{lo}{en}{hi}]"

step4_24 = prog_results.get('Step 4: + Diab, Hyp', {}).get('rr_24', {})
step4_36 = prog_results.get('Step 4: + Diab, Hyp', {}).get('rr_36', {})

summary = f"""
### NHIS pooled analysis: 4 survey years (2015{en}2018)

**Sample:** {len(pooled):,} eligible adults, {(pooled['MORTSTAT']==1).sum():,} deaths.
Self-reported liver condition: {(pooled['LIVER_EVER']==1).sum():,} (Liver+);
no liver condition: {(pooled['LIVER_EVER']==0).sum():,} (Liver{en}).

**Key findings from progressive matching:**

| Step | 24m RR (95% CI) | 36m RR (95% CI) |
|------|-----------------|-----------------|
| Crude | {fmt_rr(rr24_crude)} | {fmt_rr(rr36_crude)} |
| Step 4 (age+sex+BMI+smoking+diabetes+hypertension) | {fmt_rr(step4_24)} | {fmt_rr(step4_36)} |

**Interpretation:**
1. Self-reported liver disease in NHIS predicts elevated mortality {dash} consistent
   with the NHANES FIB-4 finding, though the exposure definition differs.
2. After matching on demographics and metabolic risk factors, the residual association
   reflects liver-disease-specific mortality risk.
3. NHIS provides ~{len(pooled)//1000}k adults (vs ~59k in NHANES pooled), with
   ~{(pooled['LIVER_EVER']==1).sum():,} exposed (vs ~1,800 with FIB-4 high).
4. The self-reported exposure captures *diagnosed* liver disease; it misses subclinical
   cases that biomarker screening would detect.

**Limitations:**
1. Self-report underestimates true liver disease prevalence
2. BMI is self-reported (vs measured in NHANES)
3. No lab covariates (no LDL-C, FPG, SBP matching)
4. PERMTH_INT approximated from death-year/quarter (not exact interview date)
5. Follow-up limited to {int(pooled['PERMTH_INT'].max())} months for 2018 cohort
"""

display(Markdown(summary))